### LIBRARY IMPORTS

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src import *
from src.data_manager import DataManager
from src.processor import Processor
from src.nn_regressor import NNRegressor
from src.gb_regressor import GBRegressor

### CONFIGURATION

In [2]:
data_manager = DataManager()

datasets_config, modeling_config = data_manager.load_config()

active_dataset = modeling_config["main"]["active_dataset"]
active_dataset_config = datasets_config[active_dataset]

problem_type = active_dataset_config["problem_type"]

gradient_boosting_config = modeling_config["gradient_boosting"]

weak_learner_key = gradient_boosting_config["weak_learner_key"]
weak_learner_config = modeling_config[weak_learner_key]

processor = Processor(**active_dataset_config)

train, valid, test = data_manager.load_processed_data()

X_train, y_train = processor.split_features_target(train)
X_valid, y_valid = processor.split_features_target(valid)
X_test, y_test = processor.split_features_target(test)

y_train, y_valid, y_test = processor.transform_target(y_train, y_valid, y_test)

### RIDGE REGRESSION

In [21]:
%%time

ridge = Ridge(alpha=0.0001, random_state=42)
ridge.fit(X_train, y_train)

ridge_valid_preds = ridge.predict(X_valid)
ridge_test_preds = ridge.predict(X_test)

print(f"Ridge validation MSE: {mean_squared_error(y_valid, ridge_valid_preds):.4f}")
print(f"Ridge validation R^2: {r2_score(y_valid, ridge_valid_preds):.4f}")
print('-' * 50)
print(f"Ridge test MSE: {mean_squared_error(y_test, ridge_test_preds):.4f}")
print(f"Ridge test R^2: {r2_score(y_test, ridge_test_preds):.4f}")

Ridge validation MSE: 0.1065
Ridge validation R^2: 0.7785
--------------------------------------------------
Ridge test MSE: 0.1060
Ridge test R^2: 0.7800
CPU times: total: 46.9 ms
Wall time: 38.1 ms


### LASSO REGRESSION

In [25]:
%%time

lasso = Lasso(alpha=0.0001, random_state=42)
lasso.fit(X_train, y_train)

lasso_valid_preds = lasso.predict(X_valid)
lasso_test_preds = lasso.predict(X_test)

print(f"Lasso validation MSE: {mean_squared_error(y_valid, lasso_valid_preds):.4f}")
print(f"Lasso validation R^2: {r2_score(y_valid, lasso_valid_preds):.4f}")
print('-' * 50)
print(f"Lasso test MSE: {mean_squared_error(y_test, lasso_test_preds):.4f}")
print(f"Lasso test R^2: {r2_score(y_test, lasso_test_preds):.4f}")

Lasso validation MSE: 0.1065
Lasso validation R^2: 0.7785
--------------------------------------------------
Lasso test MSE: 0.1060
Lasso test R^2: 0.7801
CPU times: total: 1.73 s
Wall time: 1 s


### RANDOM FOREST

In [26]:
%%time

rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X_train, y_train)

rf_valid_preds = rf.predict(X_valid)
rf_test_preds = rf.predict(X_test)

print(f"RF validation MSE: {mean_squared_error(y_valid, rf_valid_preds):.4f}")
print(f"RF validation R^2: {r2_score(y_valid, rf_valid_preds):.4f}")
print('-' * 50)
print(f"RF test MSE: {mean_squared_error(y_test, rf_test_preds):.4f}")
print(f"RF test R^2: {r2_score(y_test, rf_test_preds):.4f}")

RF validation MSE: 0.1195
RF validation R^2: 0.7515
--------------------------------------------------
RF test MSE: 0.1194
RF test R^2: 0.7522
CPU times: total: 24.9 s
Wall time: 25 s


### NEURAL NETWORK

In [24]:
class MLP(NNRegressor):
    def __init__(self, **hyperparameters):
        super().__init__(**hyperparameters)

    def fit(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_valid: np.ndarray,
        y_valid: np.ndarray, 
        patience: int = 10
    ) -> None:
        
        input_size = X_train.shape[1]
        output_size = y_train.shape[1] if len(y_train.shape) > 1 else 1
        self._get_network(input_size, output_size)
        
        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).to(torch.float32).view(-1, output_size).to(self.device)

        criterion = nn.MSELoss()
        optimizer = optim.Adam(self.parameters(), self.learning_rate)
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(self.epochs):
            self.train()
            train_loss = 0
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                loss = criterion(preds, batch_y.view_as(preds))
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()

                val_preds_np = val_preds.cpu().numpy().flatten()
                y_valid_np = y_valid_t.cpu().numpy().flatten()
                val_r2 = r2_score(y_valid_np, val_preds_np)

            print(f"Epoch: {epoch+1} | Validation MSE: {val_loss:.4f} | R^2: {val_r2:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                break
                
        if best_model:
            self.load_state_dict(best_model)

In [27]:
%%time

mlp = MLP(epochs=100, learning_rate=0.0001, hidden_size=[64, 32], batch_size=256)
mlp.fit(X_train, y_train, X_valid, y_valid)

mlp_valid_preds = mlp.predict(X_valid)
mlp_test_preds = mlp.predict(X_test)

print('-' * 50)
print(f"MLP validation MSE: {mean_squared_error(y_valid, mlp_valid_preds):.4f}")
print(f"MLP validation R^2: {r2_score(y_valid, mlp_valid_preds):.4f}")
print('-' * 50)
print(f"MLP test MSE: {mean_squared_error(y_test, mlp_test_preds):.4f}")
print(f"MLP test R^2: {r2_score(y_test, mlp_test_preds):.4f}")

Epoch: 1 | Validation MSE: 0.1131 | R^2: 0.7647
Epoch: 2 | Validation MSE: 0.1094 | R^2: 0.7725
Epoch: 3 | Validation MSE: 0.1081 | R^2: 0.7751
Epoch: 4 | Validation MSE: 0.1075 | R^2: 0.7763
Epoch: 5 | Validation MSE: 0.1074 | R^2: 0.7765
Epoch: 6 | Validation MSE: 0.1073 | R^2: 0.7769
Epoch: 7 | Validation MSE: 0.1070 | R^2: 0.7775
Epoch: 8 | Validation MSE: 0.1068 | R^2: 0.7779
Epoch: 9 | Validation MSE: 0.1066 | R^2: 0.7782
Epoch: 10 | Validation MSE: 0.1067 | R^2: 0.7781
Epoch: 11 | Validation MSE: 0.1066 | R^2: 0.7783
Epoch: 12 | Validation MSE: 0.1064 | R^2: 0.7786
Epoch: 13 | Validation MSE: 0.1065 | R^2: 0.7785
Epoch: 14 | Validation MSE: 0.1064 | R^2: 0.7787
Epoch: 15 | Validation MSE: 0.1064 | R^2: 0.7787
Epoch: 16 | Validation MSE: 0.1066 | R^2: 0.7783
Epoch: 17 | Validation MSE: 0.1066 | R^2: 0.7782
Epoch: 18 | Validation MSE: 0.1063 | R^2: 0.7788
Epoch: 19 | Validation MSE: 0.1064 | R^2: 0.7786
Epoch: 20 | Validation MSE: 0.1065 | R^2: 0.7785
Epoch: 21 | Validation MSE: 0

### GRADIENT BOOSTING (DECISION TREES)

In [3]:
gb = GBRegressor(**gradient_boosting_config, weak_learner_config=weak_learner_config)
gb.load_model("models/scores/2026_04_28_15_17/model.joblib")

gb_valid_preds = gb.predict(X_valid)
gb_test_preds = gb.predict(X_test)

print('-' * 50)
print(f"GB (DTs) validation MSE: {mean_squared_error(y_valid, gb_valid_preds):.4f}")
print(f"GB (DTs) validation R^2: {r2_score(y_valid, gb_valid_preds):.4f}")
print('-' * 50)
print(f"GB (DTs) test MSE: {mean_squared_error(y_test, gb_test_preds):.4f}")
print(f"GB (DTs) test R^2: {r2_score(y_test, gb_test_preds):.4f}")

2026-04-29 20:44:05,559 - INFO - Model loaded from models/scores/2026_04_28_15_17/model.joblib


--------------------------------------------------
GB (DTs) validation MSE: 0.1046
GB (DTs) validation R^2: 0.7825
--------------------------------------------------
GB (DTs) test MSE: 0.1044
GB (DTs) test R^2: 0.7833


### GRADIENT BOOSTING (NEURAL NETWORKS)

In [4]:
nn_gb = GBRegressor(**gradient_boosting_config, weak_learner_config=weak_learner_config)
nn_gb.load_model("models/scores/2026_04_28_15_43/model.joblib")

nn_gb_valid_preds = nn_gb.predict(X_valid)
nn_gb_test_preds = nn_gb.predict(X_test)

print('-' * 50)
print(f"GB (NNs) validation MSE: {mean_squared_error(y_valid, nn_gb_valid_preds):.4f}")
print(f"GB (NNs) validation R^2: {r2_score(y_valid, nn_gb_valid_preds):.4f}")
print('-' * 50)
print(f"GB (NNs) test MSE: {mean_squared_error(y_test, nn_gb_test_preds):.4f}")
print(f"GB (NNs) test R^2: {r2_score(y_test, nn_gb_test_preds):.4f}")

2026-04-29 20:45:08,058 - INFO - Model loaded from models/scores/2026_04_28_15_43/model.joblib


--------------------------------------------------
GB (NNs) validation MSE: 0.1061
GB (NNs) validation R^2: 0.7793
--------------------------------------------------
GB (NNs) test MSE: 0.1056
GB (NNs) test R^2: 0.7808
